## Import Libraries

In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

## Load Dataset

In [6]:
train = pd.read_csv('/Users/hemantpatidar/Desktop/Main-Minor/Summerize/datasets/samsum-train.csv')
val = pd.read_csv('/Users/hemantpatidar/Desktop/Main-Minor/Summerize/datasets/samsum-validation.csv')

In [7]:
train_data = train.sample(n = 5000, random_state = 42).reset_index( drop=True )
val_data = val.sample(n = 500, random_state = 42).reset_index(drop=True)

In [8]:
train_data.shape

(5000, 3)

## Data Preprocessing

In [9]:
import re 
def clean_data(text):
    text = re.sub(r"\r\n", " ", text)#line
    text = re.sub(r"\s+", " ", text)#space
    text = re.sub(r"\<.*?>", " ", text)#html
    text = text.strip().lower()
    return text
    

In [10]:
train_data['dialogue'] = train_data['dialogue'].apply(clean_data)
train_data['summary'] = train_data['summary'].apply(clean_data)

val_data['dialogue'] = val_data['dialogue'].apply(clean_data)
val_data['summary'] = val_data['summary'].apply(clean_data)

In [11]:
train_data.sample(10)

,id,dialogue,summary
2107,13828102,"hazel: hank, are you busy this weekend? henry:...",henry will help hazel with wallpapering on sat...
4489,13680951,"megan: can you teach me to swim? phil: wow, yo...",megan can't swim. phil will teach her at 7 on ...
1062,13728430,"katelyn: i want to visit laos, mexico, japan, ...",katelyn and eric are planing to go travelling ...
3852,13819596,nina: bonnie was in the laundry basket brian: ...,bonnie destroyed the laundry basket because sh...
4195,13821173,"emma: dear friends, i would love to thank you ...","emma thanks jo, ted, jennifer and emma for an ..."
1172,13818206,scott: kim! scott: what time does your plane a...,kimberly's plane lands at 11.45 pm. dave will ...
2796,13729349,adam: it's so boring here…. mandy: still at cl...,"adam is still at class, which he finds very bo..."
93,13819825,tom: what lines are you flying to stockholm wi...,jenny and sophia are flying to stockholm with ...
363,13730427,lona: i can't believe how mr. smith yelled at ...,mr. smith yelled at takashi who doesn't speak ...
2717,13611465,"karan: hey piyush! how are you? piyush: hey, i...",piyush is working in the security department w...


## Tokenization

In [12]:
tokenizer = T5Tokenizer.from_pretrained('t5-small')

def tokenize(data):
    inputs = tokenizer(data['dialogue'], padding = 'max_length', max_length = 512, truncation = True)
    target = tokenizer(data['summary'], padding = 'max_length', max_length = 150, truncation = True)

    inputs['labels']= target['input_ids']
    return inputs

In [13]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis = 1).tolist()

In [14]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [15]:
model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [16]:
import torch

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.backends.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print("devcie ", device)
model.to(device)

devcie  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## Model Training

In [17]:
training_arg = TrainingArguments(
    output_dir='./result',

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy='epoch',
    save_strategy='epoch',

    warmup_steps=500
    
)

In [20]:
trainer = Trainer(
    model = model,
    args= training_arg,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [21]:
trainer.train()

/Users/hemantpatidar/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,3.622340,0.370442
2,0.400131,0.354614
3,0.378415,0.348264
4,0.359021,0.346110
5,0.353899,0.345558
6,0.348943,0.345102


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/hemantpatidar/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/hemantpatidar/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/hemantpatidar/miniconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.8003815511067708, metrics={'train_runtime': 5454.7239, 'train_samples_per_second': 5.5, 'train_steps_per_second': 0.687, 'total_flos': 4060254044160000.0, 'train_loss': 0.8003815511067708, 'epoch': 6.0})

## Save model

In [26]:
model.save_pretrained('/Users/hemantpatidar/Desktop/Main-Minor/Summerize/saved_summary_model')
tokenizer.save_pretrained('/Users/hemantpatidar/Desktop/Main-Minor/Summerize/saved_summary_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/Users/hemantpatidar/Desktop/Main-Minor/Summerize/saved_summary_model/tokenizer_config.json',
 '/Users/hemantpatidar/Desktop/Main-Minor/Summerize/saved_summary_model/tokenizer.json')

In [27]:
model = T5ForConditionalGeneration.from_pretrained('./saved_summary_model')
tokenizer = T5Tokenizer.from_pretrained('./saved_summary_model')

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

## Model Evaluation

In [28]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)## clean

    #tokenize 
    inputs = tokenizer(
        dialogue,
        padding = 'max_length',
        max_length = 512,
        truncation = True,
        return_tensors = 'pt'
    ).to(device)
    ##generate summary of token 
    model.to(device)
    targets = model.generate(
        input_ids = inputs['input_ids'],
        attention_mask = inputs['attention_mask'],
        max_length = 150,
        num_beams = 4,## select best out of 4
        early_stopping = True
        
        
    )
    #token ids convert into summary 
    #decode ids
    summary = tokenizer.decode(targets[0], skip_special_tokens = True)
    return summary
    

In [29]:
summry=summarize_dialogue('''A: Hi John, are you coming to the meeting today?
B: Yes, I will be there by 10 AM.
A: Great, please bring the sales report.
B: Sure, I have already prepared it.''')
print(summry)

john will come to the meeting today at 10 am. a will bring the sales report.
